# Laboratoire SIG – SQL et statistiques descriptives en Python

## Contexte
Ce laboratoire vise à explorer un jeu de données de production primaire contenant les variables **`lat`**, **`lon`** et **`pp_open`**. Les données seront d'abord interrogées avec SQL, puis filtrées spatialement à l'aide d'un shapefile représentant une région d'intérêt.

## Objectifs
- Charger un fichier CSV dans Python
- Créer une base de données SQLite
- Écrire des requêtes SQL simples
- Convertir les données en objets spatiaux
- Extraire les points situés dans une région d'intérêt
- Calculer des statistiques descriptives simples


## 1. Importer les bibliothèques

Dans ce laboratoire, nous utilisons :
- `pandas` pour lire le CSV et manipuler les tableaux
- `sqlite3` pour créer une petite base de données SQL
- `geopandas` pour manipuler les données spatiales
- `statistics` pour calculer des statistiques descriptives


In [ ]:
import pandas as pd
import sqlite3
import geopandas as gpd
from shapely.geometry import Point
import statistics


## 2. Charger le fichier CSV

Le fichier CSV `cci_v6_2003226_ppz_takuvik_above_45n_v2.csv` se trouve dans le dossier `data/` et contenir les colonnes :
- `lat`
- `lon`
- `pp_open`


In [ ]:
df = pd.read_csv('data/cci_v6_2003226_ppz_takuvik_above_45n_v2.csv')
df.head()

### Vérification rapide du tableau

In [ ]:
print('Noms des colonnes :', list(df.columns))
print('Nombre total de lignes :', len(df))

## 3. Créer une base SQLite à partir du CSV

Nous allons enregistrer le tableau dans une base SQLite nommée `pp_database.sqlite`.


In [ ]:
conn = sqlite3.connect('pp_database.sqlite')
df.to_sql('pp_data', conn, if_exists='replace', index=False)
print('Base de données créée avec succès.')

## 4. Exploration des données avec SQL

Les prochaines cellules montrent quelques requêtes SQL simples. Lis bien chaque requête et interprète le résultat.

### Question 1 — Combien de pixels contient la base de données ?

In [ ]:
query = 'SELECT COUNT(*) AS nb_pixels FROM pp_data;'
pd.read_sql_query(query, conn)

### Question 2 — Quelle est la valeur minimale et la valeur maximale de `pp_open` ?

In [ ]:
query = '''
SELECT 
    MIN(pp_open) AS pp_min,
    MAX(pp_open) AS pp_max
FROM pp_data;
'''
pd.read_sql_query(query, conn)

### Question 3 — Quelle est la moyenne globale de `pp_open` ?

In [ ]:
query = 'SELECT AVG(pp_open) AS moyenne_globale FROM pp_data;'
pd.read_sql_query(query, conn)

### Question 4 — Combien de pixels ont une valeur de `pp_open` supérieure à 1000 ?

In [ ]:
query = 'SELECT COUNT(*) AS nb_pixels_sup_1000 FROM pp_data WHERE pp_open > 1000;'
pd.read_sql_query(query, conn)

### Question 5 — Extraire seulement les observations situées au nord de 75°N

In [ ]:
query = 'SELECT * FROM pp_data WHERE lat > 75;'
pd.read_sql_query(query, conn).head()

## 5. Convertir le tableau en points spatiaux

Chaque ligne du CSV correspond à un point défini par une longitude (`lon`) et une latitude (`lat`).


In [ ]:
geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
gdf.head()

## 6. Lire le shapefile de la région d'intérêt

Le shapefile `WDPA_WDOECM_Feb2025_Public_555637925_shp-polygons.shp` est placé dans le dossier `data/`.


In [ ]:
region = gpd.read_file('data/WDPA_WDOECM_Feb2025_Public_555637925_shp-polygons.shp')
region = region.to_crs(gdf.crs)
region.head()

## 7. Extraire les points situés dans la région

La commande suivante effectue une jointure spatiale et conserve seulement les points situés **à l'intérieur** du polygone.

In [ ]:
points_region = gpd.sjoin(gdf, region, predicate='within')
points_region.head()

### Nombre de pixels dans la région

In [ ]:
print('Nombre de pixels dans la région :', len(points_region))

## 8. Calculer des statistiques descriptives sur `pp_open` dans la région

In [ ]:
valeurs = points_region['pp_open'].dropna().tolist()

moyenne = statistics.mean(valeurs)
ecart_type = statistics.stdev(valeurs)
minimum = min(valeurs)
maximum = max(valeurs)

print('Moyenne :', moyenne)
print('Écart-type :', ecart_type)
print('Minimum :', minimum)
print('Maximum :', maximum)

## 9. Questions d'interprétation

1. Combien de pixels de production primaire se trouvent dans la région d'intérêt ?
2. La moyenne régionale de `pp_open` est-elle plus élevée ou plus faible que la moyenne globale ?
3. L'écart-type est-il faible ou élevé ? Que cela suggère-t-il sur la variabilité spatiale dans la région ?
4. Les valeurs minimales et maximales indiquent-elles une grande dispersion des données ?


## 10. Fermer la connexion à la base de données

In [ ]:
conn.close()
print('Connexion fermée.')